In [1]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# Prep

In [2]:
df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')

In [3]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])

C:\Users\hermesf\AppData\Local\Temp\ipykernel_964\996206371.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])


In [4]:
# calculate residual maturity in years
df['residual_bond_maturity'] = ((df['bond_maturity'] - df['Dates']).dt.days / 365)

In [5]:
df = df[df['residual_bond_maturity'] >= 0]

In [6]:
df['collateral_country'] = df['ISIN'].str[:2]

In [7]:
df = df[df['collateral_country'].isin(['DE','IT'])]

In [8]:
securities = tuple(df['ISIN'].unique())

In [10]:
# Data prep
query = f""" 

SELECT sum(nominal_euro) as borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}

""" 

df_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_964\463526137.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing = pd.read_sql_query(query, cnxn)


In [11]:
# Data prep
query = f""" 
 

SELECT sum(nominal_euro) as lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}

""" 

df_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_964\1481249564.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending = pd.read_sql_query(query, cnxn)


,borrowing_volume
0,1.093365e+14


In [14]:
# Data prep
query = f""" 

SELECT sum(nominal_euro) as borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}

""" 

df_borrowing_all = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_964\1113052496.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing_all = pd.read_sql_query(query, cnxn)


In [15]:
# Data prep
query = f""" 
 

SELECT sum(nominal_euro) as lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}

""" 

df_lending_all = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_964\1410661280.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending_all = pd.read_sql_query(query, cnxn)


In [16]:
share = (df_borrowing['borrowing_volume'].sum() + df_lending['lending_volume'].sum()) / (df_borrowing_all['borrowing_volume'].sum() + df_lending_all['lending_volume'].sum())

In [17]:
share

0.9074022225123722